# LangChain で作る AI Agent 実践ノートブック
## ReAct 実行・MCP 利用・Skills 利用のサンプルコード

このノートブックは講義「AI エージェント講義」のハンズオン教材です。**Google Colab 上でそのまま実行**できます。

| パート | 内容 | 講義スライドとの対応 |
|---|---|---|
| Part 1 | LangChain `create_agent` による **ReAct エージェント** | 第3部(モデル駆動ループ) |
| Part 2 | **MCP サーバー**を自作し、エージェントのツールとして接続 | 第4部(MCP) |
| Part 3 | **Skills**(SKILL.md + Progressive Disclosure)をエージェントに実装 | 第5部(Skills) |
| Part 4 | ReAct × MCP × Skills の**統合エージェント** | 全体まとめ |

### 事前準備
- **Anthropic API キー**が必要です(https://console.anthropic.com で取得)
- Colab 左側の🔑(シークレット)に `ANTHROPIC_API_KEY` という名前で登録しておくと安全です(未登録の場合は実行時に入力を求めます)
- 実行にはAPI利用料金が発生します(このノートブック全体で数円〜数十円程度)


In [ ]:
# ============================================================
# セットアップ:ライブラリのインストール
# ※ mcp は 2.x 系で langchain-mcp-adapters と互換性が壊れるため 1.x に固定
# ============================================================
%pip install -qU langchain langchain-anthropic langgraph langchain-mcp-adapters "mcp[cli]<2"

import langchain
print("langchain:", langchain.__version__)

In [ ]:
# ============================================================
# API キーの設定(Colab シークレット → なければ手入力)
# ============================================================
import os, getpass

if "ANTHROPIC_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        print("Colab シークレットから API キーを読み込みました")
    except Exception:
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key: ")

MODEL = "anthropic:claude-sonnet-4-6"   # 使用するモデル(変更可)

---
# Part 1:ReAct エージェント

**ReAct** = **Re**ason(考える)+ **Act**(ツールを使う)の交互ループ。講義スライド19の式

$$a_t \sim \pi_{\mathrm{LLM}}(a \mid x, h_t), \qquad h_{t+1} = h_t \cup \{a_t, \mathrm{obs}(a_t)\}$$

をそのまま実行します。LangChain 1.x では `create_agent` が ReAct 型ループの標準 API です(内部は LangGraph の状態グラフとして実装されています)。

ここでは Python 関数に `@tool` デコレータを付けるだけでツール化できることを確認します。**docstring がそのままモデル向けの「道具の説明書」になる**点に注目してください。

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

# ---------- ローカルツールの定義(docstring が説明書になる) ----------

PRICE_DB = {"ノートPC": 128000, "モニター": 34000, "キーボード": 9800}

@tool
def search_product_price(product: str) -> str:
    """社内カタログから商品の税抜価格(円)を検索する。product は商品名(例: ノートPC)。"""
    if product in PRICE_DB:
        return f"{product} の価格は {PRICE_DB[product]:,} 円(税抜)です。"
    return f"{product} は見つかりません。取扱商品: {', '.join(PRICE_DB)}"

@tool
def get_exchange_rate(pair: str) -> str:
    """為替レートを取得する。pair は 'USD/JPY' の形式。(デモ用の固定レート)"""
    rates = {"USD/JPY": 152.5, "EUR/JPY": 165.2}
    return f"{pair} = {rates[pair]}" if pair in rates else f"{pair} は未対応です。"

@tool
def calculate(expression: str) -> str:
    """四則演算の数式を計算する。expression は '128000 * 1.1' のような Python 式。"""
    if not set(expression) <= set("0123456789+-*/(). "):
        return "数字と + - * / ( ) . のみ使用できます。"
    return str(eval(expression))

# ---------- ReAct エージェントの作成 ----------

react_agent = create_agent(
    MODEL,
    tools=[search_product_price, get_exchange_rate, calculate],
    system_prompt="あなたは購買アシスタントです。必要に応じてツールを使い、"
                  "計算は暗算せず必ず calculate ツールで行ってください。回答は日本語で。",
)

# ---------- 実行の様子(Thought→Action→Observation)を表示するヘルパー ----------

async def run_agent(agent, query: str):
    """エージェントを実行し、各ステップ(AI の思考・ツール呼び出し・結果)を順に表示する"""
    async for step in agent.astream(
        {"messages": [{"role": "user", "content": query}]},
        stream_mode="values",
    ):
        step["messages"][-1].pretty_print()

# Colab はセル内でそのまま await が使えます
await run_agent(
    react_agent,
    "ノートPCとモニターを1台ずつ買うと税込(10%)でいくら?それは何USドル相当?",
)

### 実行結果の読み方

出力には次の3種類のメッセージが交互に現れます。これが ReAct ループの1周です。

- **Ai Message + Tool Calls**:モデルが「次にどの道具をどの引数で使うか」を決めた場面(= 行動 $a_t$ の選択)
- **Tool Message**:ツールの実行結果(= 観測 $\mathrm{obs}(a_t)$)。これが履歴 $h_t$ に追加される
- **最後の Ai Message**:ツールが不要と判断し、最終回答を出した場面

「計算は暗算せず必ずツールで」というシステムプロンプトが効き、`calculate` が呼ばれていることを確認してください。

---
# Part 2:MCP サーバーを作って接続する

講義第4部の内容を実装します。手順は2つだけです。

1. **FastMCP でサーバーを自作**(`@mcp.tool()` を付けた関数が、型ヒント+docstring から自動でツール仕様書=JSON Schema になる)
2. **`MultiServerMCPClient`** で接続し、`get_tools()` で LangChain ツールに変換してエージェントに渡す

トランスポートは **stdio**(同一マシン内でプロセスを起動して標準入出力で会話する方式)を使います。Colab のランタイム内に「社内内線」でサーバーを立てるイメージです。

In [ ]:
%%writefile weather_mcp_server.py
# ============================================================
# 自作 MCP サーバー(天気情報)— 講義スライド23と同じ FastMCP 方式
# ============================================================
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("weather")

WEATHER_DB = {
    "東京": {"weather": "晴れ", "temp_c": 34},
    "大阪": {"weather": "曇り", "temp_c": 32},
    "札幌": {"weather": "雨",   "temp_c": 24},
}

@mcp.tool()
def get_weather(city: str) -> str:
    """指定した都市の現在の天気と気温を返す。city は日本語の都市名(例: 東京)。"""
    d = WEATHER_DB.get(city)
    if d is None:
        return f"{city} のデータはありません。対応都市: {', '.join(WEATHER_DB)}"
    return f"{city} の天気は{d['weather']}、気温は {d['temp_c']}°C です。"

@mcp.tool()
def list_cities() -> str:
    """天気データが利用可能な都市の一覧を返す。"""
    return ", ".join(WEATHER_DB.keys())

if __name__ == "__main__":
    mcp.run(transport="stdio")   # 標準入出力(stdio)で待ち受け

In [ ]:
# ============================================================
# MCP クライアント:サーバーに接続し、ツールを取り込む
# ============================================================
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient({
    "weather": {                       # サーバーの呼び名
        "command": sys.executable,     # このサーバーを起動するコマンド(= python)
        "args": ["weather_mcp_server.py"],
        "transport": "stdio",
    },
    # リモートサーバーなら↓のように追加できる(複数接続可)
    # "github": {"url": "https://api.githubcopilot.com/mcp/", "transport": "http"},
})

mcp_tools = await mcp_client.get_tools()   # MCP ツール → LangChain ツールに変換

print("MCP サーバーから取得したツール:")
for t in mcp_tools:
    print(f"  - {t.name}: {t.description}")

# ---------- MCP ツールを持つエージェント ----------
mcp_agent = create_agent(
    MODEL,
    tools=mcp_tools,
    system_prompt="あなたは気象アシスタントです。天気情報は必ずツールで取得してください。回答は日本語で。",
)

await run_agent(mcp_agent, "対応している都市を確認して、いちばん涼しい都市を教えて。")

### ポイント

- エージェントのコードは Part 1 と**まったく同じ**です。ツールの出どころが「ローカル関数」から「MCP サーバー」に変わっただけで、`create_agent` から見れば同じ「道具」——これが講義で述べた **$O(M \times N) \to O(M+N)$ の標準化**の効果です。
- MCP ツールは非同期実行のため、`invoke` ではなく **`ainvoke` / `astream`** を使います(Colab はセルで直接 `await` できます)。
- デフォルトではツール呼び出しのたびに新しい MCP セッションが張られます(ステートレス)。状態を保ちたい場合は `mcp_client.session("weather")` で永続セッションを作れます。

---
# Part 3:Skills(SKILL.md)を実装する

講義第5部の **Progressive Disclosure(段階的開示)** を LangChain エージェント上に再現します。

```
skills/
├── csv-analysis/
│   ├── SKILL.md            # frontmatter(name/description)+ 手順書
│   └── scripts/analyze.py  # 決定的処理はスクリプトに切り出す
└── weekly-report/
    ├── SKILL.md
    └── template.md         # 詳細は別ファイルに分離
```

エージェントには次の**4つの道具**だけを渡します。

1. `list_skills` — 全スキルの name + description **だけ**を見る(起動時の目次)
2. `read_skill` — 必要になったスキルの SKILL.md 本文を読む
3. `read_skill_resource` — 同梱の参照ファイル(template.md 等)を読む
4. `run_skill_script` — 同梱スクリプトを実行する

「普段は目次だけ、必要なときに中身」という3段階の読み込みを、エージェント自身のツール選択として実現します。

In [ ]:
# ============================================================
# スキルフォルダとサンプルデータの作成
# ============================================================
import pathlib, textwrap

SKILLS_DIR = pathlib.Path("skills")

def write(path: str, content: str):
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(textwrap.dedent(content), encoding="utf-8")

# ---- スキル1:CSV 分析(スクリプト同梱) ----
write("skills/csv-analysis/SKILL.md", """\
    ---
    name: csv-analysis
    description: CSVファイルの集計・統計分析を行う。ユーザーがCSV、売上データ、集計、分析に言及したら使用する。
    ---
    # CSV 分析スキル

    ## 手順
    1. run_skill_script で scripts/analyze.py を実行する(引数: 対象CSVのパス)
    2. 出力された集計結果を確認し、ユーザーに分かりやすく報告する

    ## 注意
    - 集計を自分で暗算しないこと。必ずスクリプトの出力を使う
    """)

write("skills/csv-analysis/scripts/analyze.py", """\
    import csv, sys
    from collections import defaultdict

    path = sys.argv[1] if len(sys.argv) > 1 else "sales.csv"
    totals, days = defaultdict(int), set()
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            totals[row["product"]] += int(row["amount"])
            days.add(row["date"])
    print(f"対象期間: {min(days)} 〜 {max(days)}({len(days)}日間)")
    print(f"総売上: {sum(totals.values()):,} 円")
    print("製品別売上(降順):")
    for k, v in sorted(totals.items(), key=lambda x: -x[1]):
        print(f"  {k}: {v:,} 円")
    """)

# ---- スキル2:週報作成(テンプレート同梱) ----
write("skills/weekly-report/SKILL.md", """\
    ---
    name: weekly-report
    description: 社内フォーマットに沿った週報を作成する。ユーザーが週報、レポート作成、報告書に言及したら使用する。
    ---
    # 週報作成スキル

    ## 手順
    1. read_skill_resource で template.md を読み、フォーマットを確認する
    2. 与えられた事実(数値・出来事)をテンプレートに流し込む

    ## 文体ルール
    - です・ます調。1文は60文字以内
    - 数値は必ず根拠(データの出所)とセットで書く
    """)

write("skills/weekly-report/template.md", """\
    # 週報({期間})

    ## 今週のサマリ(3行以内)
    ## 実績(数値)
    ## 所感と来週の予定
    """)

# ---- サンプルデータ ----
write("sales.csv", """\
    date,product,amount
    2026-07-28,ノートPC,256000
    2026-07-28,キーボード,19600
    2026-07-29,モニター,68000
    2026-07-30,ノートPC,384000
    2026-07-31,キーボード,9800
    2026-08-01,モニター,34000
    """)

print("作成完了:")
for p in sorted(SKILLS_DIR.rglob("*")):
    if p.is_file():
        print(" ", p)

In [ ]:
# ============================================================
# Progressive Disclosure を実現する4つのツール
# ============================================================
import subprocess, sys

def _parse_frontmatter(text: str) -> dict:
    """SKILL.md 冒頭の --- で挟まれた name/description を取り出す簡易パーサ"""
    meta = {}
    if text.startswith("---"):
        for line in text.split("---")[1].strip().splitlines():
            if ":" in line and not line.startswith(" "):
                key, val = line.split(":", 1)
                meta[key.strip()] = val.strip()
            elif meta:                                  # インデント行は直前の値の続き
                meta[list(meta)[-1]] += " " + line.strip()
    return meta

@tool
def list_skills() -> str:
    """利用可能なスキルの一覧(名前と説明のみ)を返す。タスク開始時に最初に呼ぶこと。"""
    lines = []
    for d in sorted(SKILLS_DIR.iterdir()):
        f = d / "SKILL.md"
        if f.exists():
            m = _parse_frontmatter(f.read_text(encoding="utf-8"))
            lines.append(f"- {m.get('name', d.name)}: {m.get('description', '')}")
    return "\n".join(lines) or "スキルはありません。"

@tool
def read_skill(name: str) -> str:
    """指定したスキルの SKILL.md 本文(手順書)を読む。関連スキルと判断したときだけ呼ぶこと。"""
    f = SKILLS_DIR / name / "SKILL.md"
    return f.read_text(encoding="utf-8") if f.exists() else f"スキル {name} は存在しません。"

@tool
def read_skill_resource(name: str, filename: str) -> str:
    """スキルに同梱された参照ファイル(例: template.md)を読む。SKILL.md が指示した場合に呼ぶ。"""
    f = SKILLS_DIR / name / filename
    return f.read_text(encoding="utf-8") if f.exists() else f"{filename} は存在しません。"

@tool
def run_skill_script(name: str, script: str, script_args: str = "") -> str:
    """スキル同梱のスクリプトを実行し標準出力を返す。script は 'scripts/analyze.py' の形式、script_args は空白区切りの引数。"""
    f = SKILLS_DIR / name / script
    if not f.exists():
        return f"{script} は存在しません。"
    r = subprocess.run([sys.executable, str(f)] + script_args.split(),
                       capture_output=True, text=True, timeout=60)
    return r.stdout if r.returncode == 0 else f"エラー:\n{r.stderr}"

# ---------- Skills を使うエージェント ----------
skills_agent = create_agent(
    MODEL,
    tools=[list_skills, read_skill, read_skill_resource, run_skill_script],
    system_prompt=(
        "あなたは業務アシスタントです。次の手順で必ずスキルを活用してください。\n"
        "1. まず list_skills で利用可能なスキルを確認する\n"
        "2. タスクに関連するスキルがあれば read_skill で手順書を読む\n"
        "3. 手順書の指示に従い、スクリプト実行や参照ファイルの読み込みを行う\n"
        "計算・集計を自力で行わず、スキルの手順に従うこと。回答は日本語で。"
    ),
)

await run_agent(skills_agent, "sales.csv を分析して、その結果をもとに今週の週報を書いてください。")

### 実行結果の読み方

うまく動けば、ツール呼び出しが次の順に並ぶはずです。

1. `list_skills` → 目次だけ確認(**数十トークン**)
2. `read_skill("csv-analysis")` → 分析手順を取得
3. `run_skill_script(...analyze.py, "sales.csv")` → **集計はスクリプトが決定的に実行**(LLM は計算しない)
4. `read_skill("weekly-report")` → `read_skill_resource(..., "template.md")` → 書式に沿って週報を生成

「知識は普段コンテキストに載せず、必要になった瞬間だけ取りに行く」——Progressive Disclosure が、単なる**ツール設計**として実装できることが分かります。SKILL.md の description を書き換えて、スキルが選ばれたり選ばれなかったりする様子を試してみてください(トリガー精度の改善ループ=講義スライド26)。

---
# Part 4:統合 — ReAct × MCP × Skills

最後に、**ローカルツール+MCP ツール+Skills ツール**をすべて1つのエージェントに渡します。エージェントから見ればすべて同じ「道具」であり、出どころ(ローカル関数/外部サーバー/スキルフォルダ)の違いは完全に抽象化されます。

In [ ]:
# ============================================================
# 統合エージェント:3種類のツールソースを1つに
# ============================================================
all_tools = (
    [search_product_price, get_exchange_rate, calculate]          # ローカル(Part 1)
    + mcp_tools                                                   # MCP(Part 2)
    + [list_skills, read_skill, read_skill_resource, run_skill_script]  # Skills(Part 3)
)

unified_agent = create_agent(
    MODEL,
    tools=all_tools,
    system_prompt=(
        "あなたは万能業務アシスタントです。\n"
        "- タスク開始時に list_skills でスキルを確認し、関連があれば手順書に従う\n"
        "- 天気は MCP ツール、価格・為替はカタログツール、計算は calculate を使う\n"
        "- 回答は日本語で簡潔に"
    ),
)

print(f"統合エージェントのツール数: {len(all_tools)}")

await run_agent(
    unified_agent,
    "sales.csv の売上トップ製品を調べて、東京の天気と合わせて週報の冒頭あいさつ付きで報告して。",
)

---
# まとめ

| 実装したもの | 使った部品 | 講義との対応 |
|---|---|---|
| ReAct ループ | `create_agent` + `@tool` | 第3部:モデル駆動ループ $a_t \sim \pi_{\mathrm{LLM}}(a \mid x, h_t)$ |
| MCP 接続 | `FastMCP`(サーバー)+ `MultiServerMCPClient`(クライアント) | 第4部:$O(M \times N) \to O(M+N)$ の標準化 |
| Skills | SKILL.md + 4つの読み込み/実行ツール | 第5部:Progressive Disclosure |
| 統合 | 上記すべてを1エージェントに | 第6部:レイヤーの組み合わせ |

## 発展課題(演習用)

1. **Self-Consistency の実装**:同じ質問を `temperature` を上げて5回実行し、多数決を取る関数を書く(第1部)
2. **MCP サーバーの拡張**:`@mcp.resource()` でリソースを公開し、外部の公開 MCP サーバー(HTTP)にも接続してみる
3. **スキルの追加**:自分の業務手順を1つ SKILL.md 化し、description の書き方でトリガー精度がどう変わるか実験する
4. **human-in-the-loop**:`create_agent` の `interrupt_before` で、`run_skill_script` の実行前に人間の承認を挟む

## トラブルシューティング

- `ImportError: cannot import name 'RequestContext'` → **mcp 2.x が入っています**。最初のセルの `"mcp[cli]<2"` を実行し、ランタイムを再起動してください
- `401 authentication_error` → API キーが未設定です。キー設定セルを再実行してください
- MCP ツールで `NotImplementedError` → 同期 `invoke` を使っています。**`await` + `ainvoke`/`astream`** を使ってください

## 参考リンク

- LangChain Agents: https://docs.langchain.com/oss/python/langchain/agents
- langchain-mcp-adapters: https://github.com/langchain-ai/langchain-mcp-adapters
- MCP 仕様: https://modelcontextprotocol.io/
- Agent Skills: https://docs.claude.com/en/docs/agents-and-tools/agent-skills
